# Handling Large Multi-Modal Payloads in AgentCore Runtime

## Overview

This tutorial demonstrates how Amazon Bedrock AgentCore Runtime handles large payloads up to 100MB, including multi-modal content such as Excel files and images. AgentCore Runtime is designed to process rich media content and large datasets seamlessly.

### Tutorial Details

|Information| Details|
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Large Payload & Multi-Modal Processing|
| Agent type          | Single         |
| Agentic Framework   | Strands Agents |
| LLM model           | Anthropic Claude Haiku 4.5 |
| Tutorial components | Large File Processing, Image Analysis, Excel Data Processing |
| Tutorial vertical   | Data Analysis & Multi-Modal AI                                                   |
| Example complexity  | Intermediate                                                                     |
| SDK used            | Amazon BedrockAgentCore Python SDK|

### Key Features

* **Large Payload Support**: Process files up to 100MB in size
* **Multi-Modal Processing**: Handle Excel files, images, and text simultaneously
* **Data Analysis**: Extract insights from structured data and visual content
* **Base64 Encoding**: Secure transmission of binary data through JSON payloads

## Prerequisites

* Python 3.10+
* AWS credentials configured
* Docker running
* Sample Excel file and image for testing

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Create Sample Data Files

Let's create sample Excel and image files to demonstrate large payload handling:

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import os

# Create a large Excel file with sample sales data
np.random.seed(42)
data = {
    'Date': pd.date_range('2023-01-01', periods=1000, freq='h'),
    'Product': np.random.choice(['Widget A', 'Widget B', 'Widget C', 'Gadget X', 'Gadget Y'], 1000),
    'Sales': np.random.randint(1, 1000, 1000),
    'Revenue': np.random.uniform(10.0, 5000.0, 1000),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], 1000),
    'Customer_ID': np.random.randint(1000, 9999, 1000)
}

df = pd.DataFrame(data)
df.to_excel('large_sales_data.xlsx', index=False)

# Create a sample chart image
img = Image.new('RGB', (600, 500), color='white')
draw = ImageDraw.Draw(img)

# Draw a simple bar chart
products = ['Widget A', 'Widget B', 'Widget C', 'Gadget X', 'Gadget Y']
values = [250, 180, 320, 150, 280]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

max_value = max(values)
bar_width = 120
start_x = 100

for i, (product, value, color) in enumerate(zip(products, values, colors)):
    x = start_x + i * (bar_width + 20)
    height = int((value / max_value) * 400)
    y = 500 - height
    
    # Draw bar
    draw.rectangle([x, y, x + bar_width, 500], fill=color)
    
    # Add labels (simplified without font)
    draw.text((x + 10, 510), product[:8], fill='black')
    draw.text((x + 10, y - 20), str(value), fill='black')

draw.text((300, 50), 'Sales Performance by Product', fill='black')
img.save('sales_chart.png')

# Check file sizes
excel_size = os.path.getsize('large_sales_data.xlsx') / (1024 * 1024)  # MB
image_size = os.path.getsize('sales_chart.png') / (1024 * 1024)  # MB

print(f"Excel file size: {excel_size:.2f} MB")
print(f"Image file size: {image_size:.2f} MB")
print(f"Total payload size: {excel_size + image_size:.2f} MB")

## Create Multi-Modal Agent

Let's create an agent that can process both Excel files and images from large payloads:

In [ ]:
%%writefile multimodal_data_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
import pandas as pd
import base64
import io
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

# Initialize the model and agent
model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
    max_tokens=16000
)

agent = Agent(
    model=model,
    system_prompt="""
    You are a data analysis assistant that can process large Excel files and images.
    When given multi-modal data, analyze both the structured data and visual content,
    then provide comprehensive insights combining both data sources.
    """
)

@app.entrypoint
def multimodal_data_processor(payload, context):
    """
    Process large multi-modal payloads containing Excel data and images.
    
    Args:
        payload: Contains prompt, excel_data (base64), image_data (base64)
        context: Runtime context information
    
    Returns:
        str: Analysis results from both data sources
    """
    prompt = payload.get("prompt", "Analyze the provided data.")
    excel_data = payload.get("excel_data", "")
    image_data = payload.get("image_data", "")
    
    print(f"=== Large Payload Processing ===")
    print(f"Session ID: {context.session_id}")
    
    if excel_data:
        print(f"Excel data size: {len(excel_data) / 1024 / 1024:.2f} MB")
    if image_data:
        print(f"Image data size: {len(image_data) / 1024 / 1024:.2f} MB")
    print(f"Excel data {excel_data}")
    print(f"Image data {image_data}")
    print(f"=== Processing Started ===")
    # Decode base64 to bytes
    excel_bytes = base64.b64decode(excel_data)
    # Decode base64 to bytes
    image_bytes = base64.b64decode(image_data)
    
    # Enhanced prompt with data context
    enhanced_prompt = f"""{prompt}
    Please analyze both data sources and provide insights.
    """
    
    response = agent(
        [{
            "document": {
                "format": "xlsx",
                "name": "excel_data",
                "source": {
                    "bytes": excel_bytes
                }
            }
        },
        {
            "image": {
                "format": "png",
                "source": {
                    "bytes": image_bytes
                }
            }
        },
        {
            "text": enhanced_prompt
        }]
    )
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## Setup Infrastructure and Deploy Agent

In [ ]:
import boto3
import json
import zipfile
import tempfile
import os
import time

boto_session = boto3.session.Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=region)

def create_or_get_execution_role(agent_name, region, account_id):
    """Create or retrieve an IAM execution role for the AgentCore runtime."""
    iam = boto3.client('iam')
    role_name = f"AgentCoreRuntime-{agent_name[:40]}"
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account_id},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/*"}
            }
        }]
    }
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": "*"},
            {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogDelivery", "logs:PutLogEvents",
                                            "logs:CreateLogStream", "logs:DescribeLogGroups", "logs:DescribeLogStreams"], "Resource": "*"}
        ]
    }
    try:
        role = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"Execution role for AgentCore runtime {agent_name}"
        )
        iam.put_role_policy(RoleName=role_name, PolicyName="AgentCoreRuntimePermissions",
                            PolicyDocument=json.dumps(permissions_policy))
        print(f"Created IAM role: {role_name}")
        time.sleep(10)  # Allow IAM propagation
        return role['Role']['Arn']
    except iam.exceptions.EntityAlreadyExistsException:
        arn = iam.get_role(RoleName=role_name)['Role']['Arn']
        print(f"Using existing IAM role: {role_name}")
        return arn

def package_and_upload_to_s3(agent_name, files, region, account_id):
    """Package agent code as a ZIP and upload to S3 for CodeZip deployment."""
    s3 = boto3.client('s3', region_name=region)
    bucket_name = f"bedrock-agentcore-{account_id}-{region}"
    try:
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(Bucket=bucket_name,
                             CreateBucketConfiguration={'LocationConstraint': region})
        print(f"Created S3 bucket: {bucket_name}")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"Using existing S3 bucket: {bucket_name}")

    with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmpf:
        zip_path = tmpf.name
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            if os.path.exists(f):
                zf.write(f, os.path.basename(f))
                print(f"  Packaged: {f}")
    s3_key = f"{agent_name}/deployment.zip"
    s3.upload_file(zip_path, bucket_name, s3_key)
    os.unlink(zip_path)
    print(f"Uploaded to s3://{bucket_name}/{s3_key}")
    return bucket_name, s3_key

agent_name = "multimodal_data_agent"

# Create IAM execution role
role_arn = create_or_get_execution_role(agent_name, region, account_id)

# Package and upload agent code to S3
bucket_name, s3_key = package_and_upload_to_s3(
    agent_name,
    ["multimodal_data_agent.py", "requirements.txt"],
    region,
    account_id
)
print(f"\nAgent name:  {agent_name}")
print(f"Role ARN:    {role_arn}")
print(f"S3 package:  s3://{bucket_name}/{s3_key}")


In [ ]:
import time
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
status = create_response.get('status', 'CREATING')
while status not in end_status:
    time.sleep(15)
    r = agentcore_control.get_agent_runtime(agentRuntimeId=agent_runtime_id)
    status = r['status']
    print(f"Status: {status}")
print(f"\nFinal status: {status}")


## Test Large Multi-Modal Payloads

Now let's test the agent with large payloads containing both Excel data and images:

In [ ]:
import boto3
import json
from IPython.display import Markdown, display

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)

invoke_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_runtime_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How is the weather now in Athens ?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = invoke_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

try:
    events = []
    for event in invoke_response.get("response", []):
        events.append(event)
except Exception as e:
    events = [f"Error reading EventStream: {e}"]
response_text = json.loads(events[0].decode("utf-8"))
display(Markdown(response_text))


## Cleanup Resources

In [ ]:
print(f"Agent Runtime ID:  {agent_runtime_id}")
print(f"Agent Runtime ARN: {agent_runtime_arn}")


# Congratulations!

You have successfully demonstrated handling large multi-modal payloads with Amazon Bedrock AgentCore Runtime!

## What you've learned:

### Large Payload Processing
* **100MB Support**: AgentCore Runtime can handle payloads up to 100MB
* **Base64 Encoding**: Secure transmission of binary data through JSON payloads
* **Efficient Processing**: Runtime optimized for large data processing

### Multi-Modal Capabilities
* **Excel Analysis**: Processing structured data from spreadsheets
* **Image Processing**: Analyzing visual content and charts
* **Combined Analysis**: Correlating insights from multiple data types

### Key Benefits
* **Rich Data Processing**: Handle complex, multi-format datasets
* **Scalable Architecture**: Runtime designed for large workloads
* **Tool Integration**: Custom tools for specialized data processing
* **Enterprise Ready**: Secure handling of sensitive business data

This demonstrates AgentCore Runtime's capability to handle enterprise-scale data processing tasks with multiple data modalities, making it ideal for complex business intelligence and data analysis applications.